## 1. Environment Setup

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
import time
import hashlib
import json

# Add backend to path
backend_path = Path('../backend').resolve()
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

print(f"Backend path: {backend_path}")

Backend path: /home/chris/coding-task/productlens-ai/backend


## 2. Define Product Classes for CNN Training

In [2]:
# 10 product classes from CNN_Model_Train_Data.csv
# StockCodes: 22384, 22727, 22112, 23298, 20726, 21034, 21931, 22139, 22077, 22423

PRODUCT_CLASSES = [
    {"stock_code": "22384", "description": "LUNCH BAG PINK POLKADOT", "search_term": "pink polka dot lunch bag"},
    {"stock_code": "22727", "description": "ALARM CLOCK BAKELIKE RED", "search_term": "red bakelite retro alarm clock"},
    {"stock_code": "22112", "description": "CHOCOLATE HOT WATER BOTTLE", "search_term": "chocolate brown hot water bottle"},
    {"stock_code": "23298", "description": "SPOTTY BUNTING", "search_term": "spotty polka dot bunting banner"},
    {"stock_code": "20726", "description": "LUNCH BAG WOODLAND", "search_term": "woodland animal print lunch bag"},
    {"stock_code": "21034", "description": "REX CASH+CARRY JUMBO SHOPPER", "search_term": "large shopping tote bag reusable"},
    {"stock_code": "21931", "description": "JUMBO STORAGE BAG SUKI", "search_term": "large storage bag floral pattern"},
    {"stock_code": "22139", "description": "RETROSPOT TEA SET CERAMIC 11 PC", "search_term": "ceramic polka dot tea set retro"},
    {"stock_code": "22077", "description": "6 RIBBONS RUSTIC CHARM", "search_term": "rustic ribbon set craft"},
    {"stock_code": "22423", "description": "REGENCY CAKESTAND 3 TIER", "search_term": "3 tier cake stand vintage"},
]

print(f"Total product classes: {len(PRODUCT_CLASSES)}")
for i, p in enumerate(PRODUCT_CLASSES, 1):
    print(f"  {i}. [{p['stock_code']}] {p['description']}")

Total product classes: 10
  1. [22384] LUNCH BAG PINK POLKADOT
  2. [22727] ALARM CLOCK BAKELIKE RED
  3. [22112] CHOCOLATE HOT WATER BOTTLE
  4. [23298] SPOTTY BUNTING
  5. [20726] LUNCH BAG WOODLAND
  6. [21034] REX CASH+CARRY JUMBO SHOPPER
  7. [21931] JUMBO STORAGE BAG SUKI
  8. [22139] RETROSPOT TEA SET CERAMIC 11 PC
  9. [22077] 6 RIBBONS RUSTIC CHARM
  10. [22423] REGENCY CAKESTAND 3 TIER


## 3. Image Scraper Class

In [3]:
class ImageScraper:
    """Web scraper for product images using DuckDuckGo."""
    
    def __init__(self, output_dir: Path, min_size: tuple = (100, 100), max_size: tuple = (2000, 2000)):
        self.output_dir = Path(output_dir)
        self.min_size = min_size
        self.max_size = max_size
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        })
        self.downloaded_hashes = set()
    
    def search_images(self, query: str, max_results: int = 100) -> list:
        """Search for images using DuckDuckGo."""
        try:
            from duckduckgo_search import DDGS
            
            with DDGS() as ddgs:
                results = list(ddgs.images(
                    keywords=query,
                    max_results=max_results,
                    safesearch='moderate'
                ))
            
            image_urls = [r['image'] for r in results if r.get('image')]
            return image_urls
            
        except Exception as e:
            print(f"  Error searching: {e}")
            return []
    
    def download_image(self, url: str, save_path: Path, timeout: int = 10) -> bool:
        """Download and validate a single image."""
        try:
            response = self.session.get(url, timeout=timeout, stream=True)
            response.raise_for_status()
            
            # Check content type
            content_type = response.headers.get('content-type', '')
            if 'image' not in content_type:
                return False
            
            # Read image data
            image_data = response.content
            
            # Check for duplicates using hash
            image_hash = hashlib.md5(image_data).hexdigest()
            if image_hash in self.downloaded_hashes:
                return False
            
            # Validate image
            try:
                img = Image.open(BytesIO(image_data))
                img.verify()  # Verify it's a valid image
                
                # Re-open for size check
                img = Image.open(BytesIO(image_data))
                width, height = img.size
                
                # Size validation
                if width < self.min_size[0] or height < self.min_size[1]:
                    return False
                if width > self.max_size[0] or height > self.max_size[1]:
                    # Resize if too large
                    img.thumbnail(self.max_size, Image.Resampling.LANCZOS)
                
                # Convert to RGB if needed
                if img.mode in ('RGBA', 'P'):
                    img = img.convert('RGB')
                
                # Save image
                save_path.parent.mkdir(parents=True, exist_ok=True)
                img.save(save_path, 'JPEG', quality=85)
                
                self.downloaded_hashes.add(image_hash)
                return True
                
            except Exception as e:
                return False
                
        except Exception as e:
            return False
    
    def scrape_product(self, product: dict, target_count: int = 50) -> int:
        """Scrape images for a single product class."""
        stock_code = product['stock_code']
        description = product['description']
        search_term = product['search_term']
        
        # Create folder for this product
        product_dir = self.output_dir / f"{stock_code}_{description.replace(' ', '_').replace('+', '_')}"
        product_dir.mkdir(parents=True, exist_ok=True)
        
        # Check existing images
        existing = list(product_dir.glob('*.jpg'))
        if len(existing) >= target_count:
            print(f"  Already have {len(existing)} images, skipping...")
            return len(existing)
        
        print(f"  Searching for: '{search_term}'")
        
        # Search for images
        image_urls = self.search_images(search_term, max_results=target_count * 2)
        print(f"  Found {len(image_urls)} image URLs")
        
        # Download images
        downloaded = len(existing)
        for i, url in enumerate(image_urls):
            if downloaded >= target_count:
                break
            
            save_path = product_dir / f"img_{downloaded:04d}.jpg"
            
            if self.download_image(url, save_path):
                downloaded += 1
                if downloaded % 10 == 0:
                    print(f"    Downloaded {downloaded}/{target_count}")
            
            # Be nice to servers
            time.sleep(0.5)
        
        print(f"  Final count: {downloaded} images")
        return downloaded

print("ImageScraper class defined.")

ImageScraper class defined.


In [5]:
# Alternative scraper using icrawler (more reliable for batch downloads)
# Install icrawler if not available

import subprocess
try:
    from icrawler.builtin import BingImageCrawler, GoogleImageCrawler
    print("icrawler is already installed")
except ImportError:
    print("Installing icrawler...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "icrawler", "-q"])
    from icrawler.builtin import BingImageCrawler, GoogleImageCrawler
    print("icrawler installed successfully")

class ReliableImageScraper:
    """More reliable image scraper using icrawler (Bing backend)."""
    
    def __init__(self, output_dir: Path):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
    
    def scrape_product(self, product: dict, target_count: int = 50) -> int:
        """Scrape images for a single product class using Bing."""
        stock_code = product['stock_code']
        description = product['description']
        search_term = product['search_term']
        
        # Create folder for this product
        product_dir = self.output_dir / f"{stock_code}_{description.replace(' ', '_').replace('+', '_')}"
        product_dir.mkdir(parents=True, exist_ok=True)
        
        # Check existing images
        existing = list(product_dir.glob('*.jpg')) + list(product_dir.glob('*.png')) + list(product_dir.glob('*.jpeg'))
        if len(existing) >= target_count:
            print(f"  Already have {len(existing)} images, skipping...")
            return len(existing)
        
        remaining = target_count - len(existing)
        print(f"  Searching for: '{search_term}' (need {remaining} more)")
        
        # Use Bing crawler
        try:
            crawler = BingImageCrawler(
                storage={'root_dir': str(product_dir)},
                feeder_threads=1,
                parser_threads=1,
                downloader_threads=2
            )
            
            # Suppress output
            import logging
            logging.getLogger('icrawler').setLevel(logging.WARNING)
            
            crawler.crawl(
                keyword=search_term,
                max_num=remaining,
                min_size=(100, 100),
                file_idx_offset=len(existing)
            )
        except Exception as e:
            print(f"  Error with Bing crawler: {e}")
        
        # Count final images
        final_count = len(list(product_dir.glob('*.jpg')) + list(product_dir.glob('*.png')) + list(product_dir.glob('*.jpeg')))
        print(f"  Final count: {final_count} images")
        
        # Small delay between products
        time.sleep(2)
        
        return final_count

print("ReliableImageScraper class defined (using Bing backend)")


Installing icrawler...
icrawler installed successfully
ReliableImageScraper class defined (using Bing backend)


In [8]:
# Run the reliable scraper for all product classes
import logging
logging.getLogger('icrawler').setLevel(logging.ERROR)  # Suppress icrawler logs

# Re-define paths
BACKEND_PATH = Path.cwd().parent / "backend"
OUTPUT_DIR = BACKEND_PATH / "data" / "cnn_training"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

scraper = ReliableImageScraper(OUTPUT_DIR)

print("=" * 60)
print("STARTING RELIABLE IMAGE SCRAPING (Bing)")
print("=" * 60)

results = {}
for i, product in enumerate(PRODUCT_CLASSES, 1):
    print(f"\n[{i}/{len(PRODUCT_CLASSES)}] {product['description']}")
    print("-" * 40)
    
    count = scraper.scrape_product(product, target_count=50)
    results[product['stock_code']] = count

print("\n" + "=" * 60)
print("SCRAPING COMPLETE")
print("=" * 60)

total = sum(results.values())
print(f"\nTotal images scraped: {total}")
for product in PRODUCT_CLASSES:
    sc = product['stock_code']
    print(f"  {sc}: {results.get(sc, 0)} images - {product['description']}")

2025-11-27 19:36:59,427 - INFO - feeder - thread feeder-001 exit


STARTING RELIABLE IMAGE SCRAPING (Bing)

[1/10] LUNCH BAG PINK POLKADOT
----------------------------------------
  Searching for: 'pink polka dot lunch bag' (need 50 more)


2025-11-27 19:37:01,052 - INFO - parser - parsing result page https://www.bing.com/images/async?q=pink polka dot lunch bag&first=0
2025-11-27 19:37:02,852 - INFO - downloader - image #1	https://m.media-amazon.com/images/I/71A2Sn1s4rL._AC_SX679_.jpg
2025-11-27 19:37:03,702 - INFO - downloader - image #2	http://www.lolonline.ca/image/cache/data/bags/pink-polka-lunch-750x750.jpg
2025-11-27 19:37:04,615 - INFO - downloader - image #3	https://i.pinimg.com/736x/0f/9a/e4/0f9ae4b177c8e7856e2b0fb7be45a929.jpg
2025-11-27 19:37:05,348 - INFO - downloader - image #4	https://i.pinimg.com/originals/07/53/15/0753153cdfce01d04332ff3920fe8e1b.jpg
2025-11-27 19:37:05,896 - INFO - downloader - image #5	https://assets.pkimgs.com/pkimgs/rk/images/dp/wcm/202319/0129/mackenzie-pink-polka-dots-chenille-lunch-boxes-o.jpg
2025-11-27 19:37:06,573 - INFO - downloader - image #6	https://assets.pkimgs.com/pkimgs/rk/images/dp/wcm/202320/0178/mackenzie-pink-polka-dots-chenille-lunch-boxes-2-o.jpg
2025-11-27 19:37:07,

  Final count: 48 images


2025-11-27 19:37:42,491 - INFO - feeder - thread feeder-001 exit


...

  Final count: 49 images


2025-11-27 19:44:04,050 - INFO - feeder - thread feeder-001 exit



[10/10] REGENCY CAKESTAND 3 TIER
----------------------------------------
  Searching for: '3 tier cake stand vintage' (need 50 more)


2025-11-27 19:44:05,610 - INFO - parser - parsing result page https://www.bing.com/images/async?q=3 tier cake stand vintage&first=0
2025-11-27 19:44:08,223 - INFO - downloader - image #1	https://i.etsystatic.com/17630021/r/il/280daf/3175830542/il_fullxfull.3175830542_b1bm.jpg
2025-11-27 19:44:08,779 - INFO - downloader - image #2	https://i.etsystatic.com/13615175/r/il/569b1f/4556392674/il_fullxfull.4556392674_5nt5.jpg
2025-11-27 19:44:09,776 - INFO - downloader - image #3	https://i.etsystatic.com/7683639/r/il/23ec3c/4098015250/il_1080xN.4098015250_f220.jpg
2025-11-27 19:44:10,616 - INFO - downloader - image #4	https://i.etsystatic.com/13615175/r/il/29fc8f/4098678197/il_794xN.4098678197_81ix.jpg
2025-11-27 19:44:11,567 - INFO - downloader - image #5	https://www.ardenhire.co.uk/wp-content/uploads/2017/10/Vintage-3-Tier-Cake-Stand.jpg
2025-11-27 19:44:14,569 - INFO - downloader - image #6	https://i.etsystatic.com/13615175/r/il/856db2/4049198541/il_fullxfull.4049198541_1q5o.jpg
2025-11-27 

  Final count: 50 images

SCRAPING COMPLETE

Total images scraped: 464
  22384: 48 images - LUNCH BAG PINK POLKADOT
  22727: 50 images - ALARM CLOCK BAKELIKE RED
  22112: 39 images - CHOCOLATE HOT WATER BOTTLE
  23298: 50 images - SPOTTY BUNTING
  20726: 31 images - LUNCH BAG WOODLAND
  21034: 47 images - REX CASH+CARRY JUMBO SHOPPER
  21931: 50 images - JUMBO STORAGE BAG SUKI
  22139: 50 images - RETROSPOT TEA SET CERAMIC 11 PC
  22077: 49 images - 6 RIBBONS RUSTIC CHARM
  22423: 50 images - REGENCY CAKESTAND 3 TIER


In [9]:
# Verify and summarize the scraped dataset
from pathlib import Path

print("=" * 60)
print("CNN TRAINING DATASET SUMMARY")
print("=" * 60)

total_images = 0
class_distribution = []

for folder in sorted(OUTPUT_DIR.iterdir()):
    if folder.is_dir():
        images = list(folder.glob('*.jpg')) + list(folder.glob('*.png')) + list(folder.glob('*.jpeg'))
        count = len(images)
        total_images += count
        
        # Extract class name from folder
        folder_name = folder.name
        class_distribution.append({
            'folder': folder_name,
            'count': count
        })
        
        print(f"  {folder_name}: {count} images")

print("-" * 60)
print(f"TOTAL: {total_images} images across {len(class_distribution)} classes")
print(f"Average per class: {total_images / len(class_distribution):.1f} images")
print(f"Min per class: {min(c['count'] for c in class_distribution)} images")
print(f"Max per class: {max(c['count'] for c in class_distribution)} images")
print("-" * 60)
print(f"\nDataset saved to: {OUTPUT_DIR}")

CNN TRAINING DATASET SUMMARY
  20726_LUNCH_BAG_WOODLAND: 31 images
  21034_REX_CASH_CARRY_JUMBO_SHOPPER: 47 images
  21931_JUMBO_STORAGE_BAG_SUKI: 50 images
  22077_6_RIBBONS_RUSTIC_CHARM: 49 images
  22112_CHOCOLATE_HOT_WATER_BOTTLE: 39 images
  22139_RETROSPOT_TEA_SET_CERAMIC_11_PC: 50 images
  22384_LUNCH_BAG_PINK_POLKADOT: 48 images
  22423_REGENCY_CAKESTAND_3_TIER: 50 images
  22727_ALARM_CLOCK_BAKELIKE_RED: 50 images
  23298_SPOTTY_BUNTING: 50 images
------------------------------------------------------------
TOTAL: 464 images across 10 classes
Average per class: 46.4 images
Min per class: 31 images
Max per class: 50 images
------------------------------------------------------------

Dataset saved to: /home/chris/coding-task/productlens-ai/backend/data/cnn_training


## 4. Scrape Images for All Product Classes

In [4]:
# Create output directory
images_dir = backend_path / 'data' / 'images'
images_dir.mkdir(parents=True, exist_ok=True)

# Initialize scraper
scraper = ImageScraper(output_dir=images_dir)

# Scrape images for each product
TARGET_IMAGES_PER_CLASS = 50  # 50-100 as per requirements
results = {}

print("="*60)
print("STARTING IMAGE SCRAPING")
print("="*60)

for i, product in enumerate(PRODUCT_CLASSES, 1):
    print(f"\n[{i}/10] {product['description']}")
    print("-" * 40)
    
    count = scraper.scrape_product(product, target_count=TARGET_IMAGES_PER_CLASS)
    results[product['stock_code']] = count

print("\n" + "="*60)
print("SCRAPING COMPLETE")
print("="*60)
print(f"\nTotal images scraped: {sum(results.values())}")
for code, count in results.items():
    product = next(p for p in PRODUCT_CLASSES if p['stock_code'] == code)
    print(f"  {code}: {count} images - {product['description']}")

STARTING IMAGE SCRAPING

[1/10] LUNCH BAG PINK POLKADOT
----------------------------------------
  Searching for: 'pink polka dot lunch bag'
  Error searching: _get_url() https://duckduckgo.com RuntimeError: No active exception to reraise
  Found 0 image URLs
  Final count: 0 images

[2/10] ALARM CLOCK BAKELIKE RED
----------------------------------------
  Searching for: 'red bakelite retro alarm clock'
  Error searching: _get_url() https://duckduckgo.com/i.js HTTPError: HTTP Error 403: 
  Found 0 image URLs
  Final count: 0 images

[3/10] CHOCOLATE HOT WATER BOTTLE
----------------------------------------
  Searching for: 'chocolate brown hot water bottle'
  Error searching: _get_url() https://duckduckgo.com RuntimeError: No active exception to reraise
  Found 0 image URLs
  Final count: 0 images

[4/10] SPOTTY BUNTING
----------------------------------------
  Searching for: 'spotty polka dot bunting banner'
  Error searching: _get_url() https://duckduckgo.com RuntimeError: No activ

/home/chris/coding-task/.venv/lib/python3.12/site-packages/PIL/Image.py:975: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


    Downloaded 10/50
    Downloaded 20/50
    Downloaded 30/50
    Downloaded 40/50


/home/chris/coding-task/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:868: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))


    Downloaded 50/50
  Final count: 50 images

[6/10] REX CASH+CARRY JUMBO SHOPPER
----------------------------------------
  Searching for: 'large shopping tote bag reusable'
  Error searching: _get_url() https://duckduckgo.com RuntimeError: No active exception to reraise
  Found 0 image URLs
  Final count: 0 images

[7/10] JUMBO STORAGE BAG SUKI
----------------------------------------
  Searching for: 'large storage bag floral pattern'
  Error searching: _get_url() https://duckduckgo.com RuntimeError: No active exception to reraise
  Found 0 image URLs
  Final count: 0 images

[8/10] RETROSPOT TEA SET CERAMIC 11 PC
----------------------------------------
  Searching for: 'ceramic polka dot tea set retro'
  Error searching: _get_url() https://duckduckgo.com RuntimeError: No active exception to reraise
  Found 0 image URLs
  Final count: 0 images

[9/10] 6 RIBBONS RUSTIC CHARM
----------------------------------------
  Searching for: 'rustic ribbon set craft'
  Error searching: _get_

## 5. Validate Downloaded Images

In [ ]:
def validate_images(images_dir: Path) -> dict:
    """Validate all downloaded images and report statistics."""
    stats = {
        'total_images': 0,
        'valid_images': 0,
        'invalid_images': 0,
        'classes': {}
    }
    
    for class_dir in images_dir.iterdir():
        if not class_dir.is_dir():
            continue
        
        class_name = class_dir.name
        valid = 0
        invalid = 0
        
        for img_path in class_dir.glob('*.jpg'):
            try:
                img = Image.open(img_path)
                img.verify()
                valid += 1
            except:
                invalid += 1
                img_path.unlink()  # Remove corrupted image
        
        stats['classes'][class_name] = {'valid': valid, 'invalid': invalid}
        stats['total_images'] += valid + invalid
        stats['valid_images'] += valid
        stats['invalid_images'] += invalid
    
    return stats

# Validate images
print("Validating downloaded images...")
validation_stats = validate_images(images_dir)

print(f"\nValidation Results:")
print(f"  Total images: {validation_stats['total_images']}")
print(f"  Valid images: {validation_stats['valid_images']}")
print(f"  Invalid (removed): {validation_stats['invalid_images']}")
print(f"\nPer-class breakdown:")
for class_name, counts in validation_stats['classes'].items():
    print(f"  {class_name}: {counts['valid']} valid")

## 6. Summary

In [ ]:
print("="*60)
print("IMAGE SCRAPING SUMMARY")
print("="*60)
print(f"\nImages directory: {images_dir}")
print(f"Product classes: {len(PRODUCT_CLASSES)}")

# Count final images
total = 0
for class_dir in images_dir.iterdir():
    if class_dir.is_dir():
        count = len(list(class_dir.glob('*.jpg')))
        total += count
        print(f"  {class_dir.name}: {count} images")

print(f"\nTotal images: {total}")
print(f"Average per class: {total / len(PRODUCT_CLASSES):.1f}")
print(f"\nNext: Run 04_cnn_training.ipynb to train the model")